# Playing with Sparsity

What we are doing here is trying to see whether we can reasonably hope to store sparse tensors containing detailed incidence structures for a large hypergraph. In this case, we'll store a 3D tensor `S` where `S[i,j,k] = 1` if nodes `i` and `j` are both in hyperedge `k`, and zero otherwise. This tensor has shape `(m, m, n)` where `m` is the number of hyperedges and `n` is the number of nodes.

In [1]:
import xgi
import torch
import sys

In [2]:
H = xgi.load_xgi_data("senate-bills")

n = len(H.nodes)
m = len(H.edges)

Here's our main loop to construct a list of 3-indices where the tensor should have nonzero entries. Note that there's a hardcoded limit of checking 1% of edges (like we might do in a batch of stochastic gradient descent), which would need to be removed for a full test. 

In [3]:
batch_size = int(m / 100)

ix = []

for i, e in enumerate(H.edges): 
    
    # hardcoded in for testing
    if i >= batch_size: 
        break 
    
    if i % 1000 == 0:
        print(f"Processing edge {i} of {m}")
    # neighbors = H.edges.neighbors(e)
    
    for f in H.edges: 
        if f == e: 
            continue
        
        intersection = set(H.edges.members(e)) & set(H.edges.members(f))
        
        if len(intersection) != 0: 
            for v in intersection: 
                ix.append((int(e), int(f), int(v)))

Processing edge 0 of 29157


Having done the above, we now have a list of indices of the form `(e, f, v)` where `e` and `f` are hyperedge indices and v is a node index. We would *like* to form a 3d sparse tensor with shape `(m, m, n)` to hold these indices, but for some reason PyTorch does not support sparse tensors of more than 2 dimensions.

So, we need to do something kinda dumb: instead of a (m, m, n) tensor, we will make a (m, m*n) tensor, where the second index is formed by "flattening" the (e, f) pair into a single index of the form `e*m+f`.

In [4]:
# new second index is f*n + v, with mn total entries
# new shape is (m, mn)

wrapped_ix = [(e*m+f, v) for (e, f, v) in ix]

Now we are ready to make our dumb lil sparse tensor. 

In [5]:
s = torch.sparse_coo_tensor(
    indices=torch.tensor(wrapped_ix).T,
    values=torch.ones(len(wrapped_ix)),
    size=(m*m, n), check_invariants=False, requires_grad=False
)

Here's code to check the size of this tensor on disk, in MB: 

In [6]:
s.element_size()*s._nnz() / 1e6  # in MB

5.949612

That's 1% the size we'd need, so multiply by 100 to get an estimate of the size of the full sparse tensor we need to form. 

The point of doing this whole thing is that now we can form counts of things like "estimate of number of nodes of size 1 contained in the `e`-`f` pair" by matrix multiplication: 

In [7]:
z = torch.randn((n,), requires_grad=True) # some random labels

counts = s@z # e-f entry of counts is sum over v of s[e,f,v] * z[v], which is an estimate of the number of nodes in edge e and f of label 1

In [8]:
# need to check but I *think* this should be right if one prefers to work with a 2-d tensor of counts

counts = counts.reshape((m, m))

Now we can use cout for whatever we need next in our pipeline. Here's a quick example just to check that we can backprop to get gradients in $z$. 

In [9]:
loss = (counts**2).mean()
loss.backward()

The thing that appears to be very cool about this is that the forward and backward step here are not exactly fast, but they *appear* to scale very well, sublinearly with batch size. So, although there's a lot of stuff here which is quite slow, once we have formed the relevant sparse tensors, we should be able to move through even large batches very quickly. 

In [10]:
z.grad

tensor([ 0.0000e+00,  8.0317e-06,  2.3681e-06, -6.1904e-06,  4.7812e-05,
         2.3606e-04, -2.2192e-04, -2.4464e-05,  1.6237e-05, -1.2128e-05,
        -7.3223e-06,  2.2271e-06, -1.0291e-04, -1.7320e-07, -5.8816e-06,
         1.5235e-06, -7.6234e-05, -3.7966e-05,  1.2796e-04, -5.2287e-05,
         3.5059e-05, -1.3689e-06,  6.2280e-05,  2.9904e-05,  4.4116e-05,
         3.5964e-06, -7.4025e-05,  3.5957e-06,  4.1633e-05, -2.7686e-05,
         3.5225e-06, -2.1410e-05, -2.0164e-06, -3.2522e-06, -1.2267e-04,
        -2.1040e-05,  3.4976e-05,  1.0362e-05, -9.9061e-05,  5.7078e-05,
         1.2624e-05,  2.9578e-05,  1.6961e-06, -1.1407e-06,  8.4612e-06,
         9.2509e-06,  6.2148e-06, -1.4616e-06, -1.2830e-05, -4.0558e-06,
         7.3568e-05,  2.0732e-05,  3.8415e-05,  3.6775e-05, -4.6665e-07,
        -1.7053e-06, -1.0302e-05,  1.0268e-04,  3.7810e-07,  2.2864e-05,
         6.9704e-06, -8.3556e-06,  7.0005e-05, -4.7855e-05, -4.3716e-06,
         3.4784e-05,  2.5183e-05,  3.9864e-06, -1.5

Sweet, delicious gradients! 